In [1]:
from concurrent.futures import ThreadPoolExecutor

import geopandas as gpd
import rasterio as rio

In [2]:
DATA_PREFIX = "../data"
OUTPUT_PREFIX = "../output"

TRAIN_PARQUET = f"{OUTPUT_PREFIX}/train_only_biomass.parquet"
TEST_PARQUET = f"{OUTPUT_PREFIX}/test_only_biomass.parquet"
EXTRACT_TRAIN_PARQUET = f"{OUTPUT_PREFIX}/extract_train.parquet"
EXTRACT_TEST_PARQUET = f"{OUTPUT_PREFIX}/extract_test.parquet"
SUBMISSION_CSV = f"{DATA_PREFIX}/sample_submission.csv"

In [3]:
BANDS_S2 = ["BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"]
BANDS_S2_DIST = [f"{b}_DIST" for b in BANDS_S2]
BANDS_S1 = ["VV", "VH"]
BANDS_S1_DIST = [f"{b}_DIST" for b in BANDS_S1]

In [4]:
train_df = gpd.read_parquet(TRAIN_PARQUET)
train_df

,x,y,year,biomass,tile_id,geometry
0,3137124.7416963745,1810803.9191588927,2020,28.742064,038052,POINT (-3.54712 38.43217)
1,3138084.7416963745,1808543.9191588927,2020,61.379017,038052,POINT (-3.53221 38.41357)
2,3136834.7416963745,1809903.9191588927,2020,41.445705,038052,POINT (-3.54874 38.42368)
3,3137594.7416963745,1810823.9191588927,2020,25.911734,038052,POINT (-3.54186 38.4331)
4,3137914.7416963745,1808923.9191588927,2020,40.892021,038052,POINT (-3.53481 38.41668)
...,...,...,...,...,...,...
5148499,3111664.7416963745,2243933.9191588927,2019,58.634174,035805,POINT (-4.70534 42.23312)
5148500,3111624.7416963745,2243883.9191588927,2019,60.487827,035805,POINT (-4.70571 42.23261)
5148501,3113104.7416963745,2243803.9191588927,2019,118.166924,035805,POINT (-4.68791 42.2345)
5148502,3113074.7416963745,2243833.9191588927,2019,104.021767,035805,POINT (-4.68834 42.23471)


In [5]:
def run_per_tile(index, df, tile_ids):
    tile_id = tile_ids[index]
    tile_sample = df[df["tile_id"] == tile_id]
    coords = [coord for coord in zip(tile_sample.geometry.x, tile_sample.geometry.y)]
    years = tile_sample["year"].unique()

    for year in years:
        print(f"Run {tile_id} {year} {index + 1} / {len(tile_ids)}")

        df_mask = (df["tile_id"] == tile_id) & (df["year"] == year)

        with rio.open(
            f"gs://gee-ramiqcom-s4g-bucket/opengeohub_summerschool_2026/s2/tile_{tile_id}_{year}_S2_L2A_composite_{year}-06-01_{year}-07-31_10m.tif"
        ) as src:
            df.loc[df_mask, BANDS_S2] = [data for data in src.sample(coords)]

        with rio.open(
            f"gs://gee-ramiqcom-s4g-bucket/opengeohub_summerschool_2026/s2_distance/tile_{tile_id}_{year}_S2_L2A_composite_{year}-06-01_{year}-07-31_10m.tif"
        ) as src:
            df.loc[
                df_mask,
                BANDS_S2_DIST,
            ] = [data for data in src.sample(coords)]

        with rio.open(
            f"gs://gee-ramiqcom-s4g-bucket/opengeohub_summerschool_2026/s1/tile_{tile_id}_{year}_S1_RTC_composite_{year}-06-01_{year}-07-31_10m.tif"
        ) as src:
            df.loc[df_mask, BANDS_S1] = [data for data in src.sample(coords)]

        with rio.open(
            f"gs://gee-ramiqcom-s4g-bucket/opengeohub_summerschool_2026/s1_distance/tile_{tile_id}_{year}_S1_RTC_composite_{year}-06-01_{year}-07-31_10m.tif"
        ) as src:
            df.loc[
                df_mask,
                BANDS_S1_DIST,
            ] = [data for data in src.sample(coords)]

In [ ]:
tile_ids = train_df["tile_id"].unique()

with ThreadPoolExecutor(8) as executor:
    jobs = []
    for index in range(len(tile_ids)):
        jobs.append(executor.submit(run_per_tile, index, train_df, tile_ids))
    for job in jobs:
        try:
            job.result()
        except Exception as e:
            print(f"Error: {e}")

train_df

Run 036385 2019 3 / 178
Run 026261 2021 5 / 178
Run 030430 2019 7 / 178
Run 051927 2019 8 / 178
Run 038052 2020 1 / 178
Run 086895 2016 2 / 178
Run 039974 2019 6 / 178
Run 038510 2019 4 / 178
Run 045997 2019 9 / 178
Run 088988 2016 10 / 178
Run 085676 2016 11 / 178
Run 034493 2019 12 / 178
Run 040151 2019 13 / 178
Run 083291 2016 14 / 178
Run 029528 2019 15 / 178
Run 034594 2019 16 / 178
Run 042237 2019 17 / 178
Run 045356 2019 18 / 178
Run 045312 2019 19 / 178
Run 026863 2021 20 / 178
Run 092872 2016 21 / 178
Run 041342 2019 22 / 178
Run 041182 2019 23 / 178
Run 098252 2016 24 / 178
Run 051637 2019 25 / 178
Run 040307 2019 26 / 178
Run 061154 2020 27 / 178
Run 045647 2019 28 / 178
Run 044463 2019 29 / 178
Run 027166 2021 30 / 178


In [ ]:
# to parquet
train_df.to_parquet(EXTRACT_TRAIN_PARQUET)

In [ ]:
test_df = gpd.read_parquet(TEST_PARQUET)
test_df

,row_id,tile_id,x,y,year,geometry
0,028631_3040064_2251003,028631,3040064.7416963745,2251003.9191588927,2021,POINT (-5.57242 42.1662)
1,028631_3041214_2252063,028631,3041214.7416963745,2252063.9191588927,2021,POINT (-5.56121 42.17768)
2,028631_3041184_2251693,028631,3041184.7416963745,2251693.9191588927,2021,POINT (-5.56071 42.17436)
3,028631_3039864_2250783,028631,3039864.7416963745,2250783.9191588927,2021,POINT (-5.57428 42.16389)
4,028631_3040224_2251483,028631,3040224.7416963745,2251483.9191588927,2021,POINT (-5.57163 42.17073)
...,...,...,...,...,...,...
45014,061751_3374644_2043173,061751,3374644.7416963745,2043173.9191588927,2020,POINT (-1.21909 40.85472)
45015,061751_3374594_2043203,061751,3374594.7416963745,2043203.9191588927,2020,POINT (-1.21972 40.85492)
45016,061751_3374624_2043173,061751,3374624.7416963745,2043173.9191588927,2020,POINT (-1.21932 40.85469)
45017,061751_3374614_2043223,061751,3374614.7416963745,2043223.9191588927,2020,POINT (-1.21952 40.85512)


In [ ]:
tile_ids = test_df["tile_id"].unique()

with ThreadPoolExecutor(8) as executor:
    jobs = []
    for index in range(len(tile_ids)):
        jobs.append(executor.submit(run_per_tile, index, test_df, tile_ids))
    for job in jobs:
        try:
            job.result()
        except Exception as e:
            print(f"Error: {e}")

test_df

Run S2 031028 2019 6 / 10Run S2 045101 2019 2 / 10

Run S2 060543 2020 7 / 10
Run S2 035814 2019 4 / 10
Run S2 047474 2019 8 / 10
Run S2 062333 2020 3 / 10
Run S2 028631 2021 1 / 10
Run S2 062332 2020 5 / 10
Run S1 045101 2019 2 / 10
Run S2 061751 2020 9 / 10
Run S1 060543 2020 7 / 10
Run S1 047474 2019 8 / 10
Run S2 040897 2019 10 / 10
Run S1 061751 2020 9 / 10
Run S1 031028 2019 6 / 10
Run S1 040897 2019 10 / 10
Run S1 062332 2020 5 / 10
Run S1 035814 2019 4 / 10
Run S1 062333 2020 3 / 10
Run S1 028631 2021 1 / 10


,row_id,tile_id,x,y,year,geometry,BLUE,GREEN,RED,NIR,SWIR1,SWIR2,VV,VH
0,028631_3040064_2251003,028631,3040064.7416963745,2251003.9191588927,2021,POINT (-5.57242 42.1662),147.0,318.0,152.0,3368.0,1293.0,561.0,2425.0,548.0
1,028631_3041214_2252063,028631,3041214.7416963745,2252063.9191588927,2021,POINT (-5.56121 42.17768),155.0,318.0,167.0,3011.0,1256.0,539.0,1591.0,403.0
2,028631_3041184_2251693,028631,3041184.7416963745,2251693.9191588927,2021,POINT (-5.56071 42.17436),159.0,330.0,148.0,4007.0,1171.0,486.0,1360.0,447.0
3,028631_3039864_2250783,028631,3039864.7416963745,2250783.9191588927,2021,POINT (-5.57428 42.16389),163.0,329.0,167.0,3221.0,1318.0,559.0,1893.0,504.0
4,028631_3040224_2251483,028631,3040224.7416963745,2251483.9191588927,2021,POINT (-5.57163 42.17073),139.0,276.0,123.0,4183.0,1180.0,431.0,1943.0,350.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45014,061751_3374644_2043173,061751,3374644.7416963745,2043173.9191588927,2020,POINT (-1.21909 40.85472),289.0,583.0,396.0,4034.0,2086.0,1174.0,1288.0,360.0
45015,061751_3374594_2043203,061751,3374594.7416963745,2043203.9191588927,2020,POINT (-1.21972 40.85492),686.0,1111.0,1201.0,4128.0,2865.0,2012.0,1127.0,282.0
45016,061751_3374624_2043173,061751,3374624.7416963745,2043173.9191588927,2020,POINT (-1.21932 40.85469),286.0,574.0,375.0,3594.0,2122.0,1218.0,1332.0,356.0
45017,061751_3374614_2043223,061751,3374614.7416963745,2043223.9191588927,2020,POINT (-1.21952 40.85512),210.0,405.0,271.0,3095.0,1383.0,667.0,1321.0,381.0


In [ ]:
# to parquet
test_df.to_parquet(EXTRACT_TEST_PARQUET)